#Installation

In [1]:
!pip install git+https://github.com/EMI-Group/tensorneat.git

  Cloning https://github.com/EMI-Group/tensorneat.git to /tmp/pip-req-build-3qzvfgcj
  Running command git clone --filter=blob:none --quiet https://github.com/EMI-Group/tensorneat.git /tmp/pip-req-build-3qzvfgcj
  Resolved https://github.com/EMI-Group/tensorneat.git to commit b9778dcf8b2db5f3698b3965073c2181608c467c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.1/344.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.6/86.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 136.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 27.0 MB/s e

In [2]:
# !pip install -U "jax[tpu]==0.4.28"

In [3]:
!pip install evojax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.8/303.8 kB 14.4 MB/s eta 0:00:00


In [4]:
!pip install opencv-python

#Libraries

In [5]:
# Import necessary modules
import jax
import jax.numpy as jnp
import argparse
import os
from jax import random
from evojax.task.slimevolley import SlimeVolley
from tensorneat.problem import BaseProblem
from tensorneat.pipeline import Pipeline
from tensorneat.algorithm.neat import NEAT
from tensorneat.genome import DefaultGenome, BiasNode, DefaultConn,DefaultMutation
from tensorneat.genome.operations import DefaultCrossover, DefaultDistance, DefaultMutation
from tensorneat.common import ACT, AGG
from typing import Tuple
from functools import partial

#SlimeVolleyBall Environment

In [6]:
import jax.numpy as jnp

def adjusted_fitness(s_fitness):
    """
    Calculates the total adjusted fitness for a species, as described in the original NEAT paper.

    Args:
        s_fitness (jnp.ndarray): A JAX array containing the fitness values of individuals
                                 in a single species. Non-members are represented by -jnp.inf.

    Returns:
        jnp.ndarray: The total adjusted fitness of the species.
    """
    # 1. Create a boolean mask to identify valid fitness values (i.e., not -jnp.inf)
    is_member_mask = jnp.isfinite(s_fitness)

    # 2. Count the number of members in the species
    species_size = jnp.sum(is_member_mask)

    # Use a conditional to handle the case of an empty species (size=0) to avoid division by zero.
    species_size = jnp.where(species_size == 0, 1, species_size)

    # 3. Adjust the fitness of each member by dividing by the species size.
    # The where clause ensures only valid fitnesses are adjusted.
    adjusted_s_fitness = jnp.where(is_member_mask, s_fitness / species_size, 0.0)

    # 4. Sum the adjusted fitness values to get the species' total adjusted fitness.
    total_adjusted_fitness = jnp.sum(adjusted_s_fitness)

    return total_adjusted_fitness

In [7]:
class SlimeVolleyEnv(BaseProblem):
    jitable = True  # Needed for TensorNEAT JIT support

    def __init__(self, max_episode_steps=3000, num_episodes=3):
        super().__init__()
        self.max_steps = max_episode_steps
        self.episodes = num_episodes

        # Create task instance
        self.task = SlimeVolley(test=False, max_steps=self.max_steps) #To imitate the same conditions as testing phase.
        self.test_task = SlimeVolley(test=True, max_steps=3000)

        # JIT compile reset and step functions
        self.task_reset = jax.jit(self.task.reset)
        self.task_step = jax.jit(self.task.step)
        self.test_task_reset = jax.jit(self.test_task.reset)
        self.test_task_step = jax.jit(self.test_task.step)

    def calculate_step_reward(self,prev_state, current_state, reward,current_step, max_steps=3000):
        """
        Step-wise reward function for NEAT training on SlimeVolleyball.
        Calculates reward based on life changes between consecutive states.

        Args:
            prev_state: Previous game state (with agent_left and agent_right life values)
            current_state: Current game state (with agent_left and agent_right life values)
            current_step: Current timestep (0 to max_steps)
            max_steps: Maximum steps per episode (default 3000)

        Returns:
            Scalar reward for this step

        Reward structure:
            +100: Opponent loses a life (good!)
            -200: Agent loses a life (bad!)
            -0.1: Small step penalty (encourages efficiency)
            +500 to +0: Victory time bonus (earlier win = more bonus)
        """
        # Get current lives
        prev_agent_life = prev_state.game_state.agent_right.life
        prev_opp_life = prev_state.game_state.agent_left.life

        curr_agent_life = current_state.game_state.agent_right.life
        curr_opp_life = current_state.game_state.agent_left.life

        # Detect life changes
        agent_lost_life = reward<0
        opponent_lost_life = reward>0

        # Check if this is the finishing blow (opponent goes from 1 -> 0 lives)
        finishing_blow = (prev_opp_life == 1) & (curr_opp_life == 0)

        # Calculate time bonus for finishing blow
        # Bonus decreases linearly from 500 (step 0) to 0 (step 3000)
        # Formula: 500 * (1 - current_step / max_steps)
        time_bonus = jnp.where(
            finishing_blow,
            500.0 * (1.0 - current_step / max_steps),
            0.0
        )

        # Calculate base step reward
        base_reward = jnp.where(
            opponent_lost_life,
            100.0,  # Positive reward for damaging opponent
            jnp.where(
                agent_lost_life,
                -200.0,  # Negative reward for losing life
                -0.1  # Small penalty for each step without scoring
            )
        )
        condition1 = (curr_opp_life == 3) &(prev_opp_life==4)
        condition2 = (curr_opp_life == 2) &(prev_opp_life==3)
        condition3 = (curr_opp_life == 1) &(prev_opp_life==2)
        # bonus_reward  = jnp.where(condition1,300,jnp.where(condition2,400,jnp.where(condition3,500,0)))
        bonus_reward  = jnp.where(condition1,150,jnp.where(condition2,250,jnp.where(condition3,350,0)))
        # Total reward = base reward + time bonus
        reward = base_reward + time_bonus + bonus_reward

        return reward






    def evaluate(self, state, randkey, act_func, params):
        """Evaluate with episode-end reward calculation, maintaining JAX parallelization"""

        def run_episode(ep_key):
            # Reset environment
            reset_key = ep_key[None, :]
            task_state = self.task_reset(reset_key)

            # Carry: (task_state, total_reward, randkey)
            sum_reward = jnp.array(0.0, dtype=jnp.float32)
            init_carry = (task_state, sum_reward)

            def step_fn(carry, _):
                task_state, total_reward = carry

                # Get observation and action
                obs = task_state.obs.squeeze()
                action_out = act_func(task_state, params, obs)
                action = (action_out > 0.0).astype(jnp.float32)
                jax_action = action[None,:]

                # Step environment
                new_state, reward, done = self.task_step(task_state, jax_action)
                # done = jnp.squeeze(done).astype(jnp.bool_)
                reward = jnp.squeeze(reward).astype(jnp.float32)
                step_reward = self.calculate_step_reward(task_state,new_state,reward,new_state.steps)
                step_reward = jnp.squeeze(step_reward).astype(jnp.float32)
                # reward = jnp.where(reward > 0, reward.dtype.type(2.0), reward)reward = jnp.where(reward>0,reward.)
                total_reward = total_reward + step_reward

                return (new_state, total_reward), None
                # # CRITICAL: Use previous 'ended' flag for freezing decision
                # # This ensures we capture the state when done=True fires
                # final_state = jax.tree_util.tree_map(
                #     lambda old, new: jnp.where(ended, old, new),
                #     task_state, new_state
                # )

                # # Update ended flag AFTER freezing decision
                # new_ended = ended | done

                # return (final_state, new_ended), None

            # Run episode for max_steps (state freezes after done=True)
            final_carry, _ = jax.lax.scan(step_fn, init_carry, None, length=self.max_steps)
            return final_carry[1]  # total_reward

        # Run multiple episodes and average reward
        ep_keys = jax.random.split(randkey, self.episodes)
        rewards = jax.vmap(run_episode)(ep_keys)
        return jnp.mean(rewards)

    @property
    def input_shape(self):
        return self.task.obs_shape  # Provided by SlimeVolley

    @property
    def output_shape(self):
        return self.task.act_shape  # Provided by SlimeVolley

    def squeeze_state(self, state):
        return jax.tree_util.tree_map(
            lambda x: jnp.squeeze(x, axis=0) if hasattr(x, "ndim") and x.ndim > 0 else x,
            state
        )

    def show(self, state, randkey, act_func, params, *args, **kwargs):
        """Render a few episodes to visually inspect performance."""
        import time
        import numpy as np
        print("\n" + "="*50)
        print("SLIMEVOLLEY AGENT PERFORMANCE")
        print("="*50)

        best_reward = float('-inf')
        best_screens = []
        best_opp_life = float('inf')  # Track opponent life (lower is better)
        best_steps = 0  # Track episode length (higher is better)

        # JAX-compiled function for running a single episode
        @jax.jit
        def run_single_episode(episode_key):
            """Run a single episode and return states, rewards, and done flags."""
            test_task_state = self.test_task_reset(episode_key[None, :])

            def step_fn(carry, _):
                task_state, total_reward, step_count, terminated = carry

                obs = task_state.obs.squeeze()
                action_out = act_func(task_state, params, obs)
                action = (action_out > 0.0).astype(jnp.float32)
                jax_action = action[None, :]

                new_state, reward, done = self.test_task_step(task_state, jax_action)
                reward = jnp.squeeze(reward).astype(jnp.float32)
                done = jnp.squeeze(done).astype(jnp.bool_)

                # Update only if not already terminated
                new_total_reward = jnp.where(terminated, total_reward, total_reward + reward)
                new_step_count = jnp.where(terminated, step_count, step_count + 1)
                new_terminated = terminated | done

                # Use previous state if already terminated
                final_state = jax.tree_util.tree_map(
                    lambda old, new: jnp.where(terminated, old, new),
                    task_state, new_state
                )

                return (final_state, new_total_reward, new_step_count, new_terminated), {
                    'state': final_state,
                    'reward': reward,
                    'done': done,
                    'terminated': new_terminated
                }

            init_carry = (test_task_state, jnp.array(0.0, dtype=jnp.float32),
                        jnp.array(0, dtype=jnp.int32), jnp.array(False))

            final_carry, history = jax.lax.scan(step_fn, init_carry, None, length=3000)

            return {
                'final_reward': final_carry[1],
                'final_steps': final_carry[2],
                'states': history['state'],
                'rewards': history['reward'],
                'dones': history['done'],
                'terminated_flags': history['terminated']
            }

        # Run episodes using JAX
        episode_keys = jax.random.split(randkey, 5)

        for i in range(5):
            # Run episode with JAX
            episode_result = run_single_episode(episode_keys[i])

            total_reward = float(episode_result['final_reward'])
            final_steps = int(episode_result['final_steps'])
            states = episode_result['states']
            dones = episode_result['dones']

            # Find the actual end of the episode
            done_indices = jnp.where(dones)[0]
            if len(done_indices) > 0:
                actual_end_step = int(done_indices[0]) + 1
                # agent_left is the opponent, agent_right is our agent
                # Extract life value properly - handle potential array dimensions
                opp_life_array = states.game_state.agent_left.life[done_indices[0]]
                # Squeeze any extra dimensions and convert to scalar
                opp_end_life = float(jnp.squeeze(opp_life_array))
            else:
                actual_end_step = 3000
                opp_end_life = 5.0  # Max life if episode didn't end

            print(f"Episode {i+1} - Reward: {total_reward:.2f}, Steps: {actual_end_step}, Opponent Life: {opp_end_life}")

            # Generate screens for this episode (only up to actual end)
            current_screens = []
            for step in range(actual_end_step):
                step_state = jax.tree_util.tree_map(lambda x: x[step], states)
                render_state = self.squeeze_state(step_state)
                current_screens.append(self.test_task.render(render_state))

            # Determine if this episode is better based on hierarchical criteria
            is_better = False

            if total_reward > best_reward:
                # Higher reward is always better
                is_better = True
            elif total_reward == best_reward:
                # Same reward: prefer lower opponent life
                if opp_end_life < best_opp_life:
                    is_better = True
                elif opp_end_life == best_opp_life:
                    # Same reward and opponent life: prefer longer episode
                    if actual_end_step > best_steps:
                        is_better = True

            if is_better:
                best_reward = total_reward
                best_opp_life = opp_end_life
                best_steps = actual_end_step
                best_screens = current_screens.copy()
                print(f"  -> New best episode!")

        print(f"\n{'='*50}")
        print(f"BEST EPISODE SUMMARY:")
        print(f"  Reward: {best_reward:.2f}")
        print(f"  Opponent Life Remaining: {best_opp_life}")
        print(f"  Episode Length: {best_steps} steps")
        print(f"{'='*50}\n")

        # Create GIF only from the best episode
        log_dir = './'
        gif_file = os.path.join(log_dir, 'slimevolley_best.gif')
        if best_screens:
            best_screens[0].save(gif_file, save_all=True, append_images=best_screens[1:],
                                duration=40, loop=0)
            print(f"GIF created from best episode and saved to: {gif_file}")
        else:
            print("No screens to save for GIF")

#Training

In [8]:
neatalgorithm=NEAT(
        pop_size=400,  # Larger population for complex task
        species_size=200,  # More species for diversity
        survival_threshold=0.2,  # Top 30% survive
        genome=DefaultGenome(
            num_inputs=12,  # SlimeVolley observation space
            num_outputs=3,  # SlimeVolley action space (3 discrete actions)
            max_nodes=100,
            max_conns=1500,
            init_hidden_layers=(),  # Start with no hidden layers
            node_gene=BiasNode( bias_init_mean= 0.0,
                                bias_init_std = 0.1,
                                bias_mutate_power = 0.05,
                                bias_mutate_rate = 0.1,
                                bias_replace_rate = 0.005,
                                bias_lower_bound = -3,
                                bias_upper_bound = 3,
                                aggregation_default = AGG.sum,
                                aggregation_options = [AGG.sum, AGG.mean, AGG.max,],
                                aggregation_replace_rate = 0.0,
                                activation_default = ACT.relu,
                                activation_options=[
                                                    ACT.relu,
                                                    ACT.sigmoid,
                                                  # ACT.identity,  # Can help with periodic behaviors
                                                  ],
                                activation_replace_rate= 0.05,


                                ),
            conn_gene=DefaultConn(weight_init_mean  = 0.0,
                                  weight_init_std   = 1.0,
                                  weight_mutate_power = 0.2, #scale of perturbation
                                  weight_mutate_rate  = 0.1,  #probability of perturbation
                                  weight_replace_rate = 0.005,
                                  weight_lower_bound = -5,
                                  weight_upper_bound = 5,
                                  ),
            mutation=DefaultMutation(conn_add = 0.3,
                                     conn_delete  = 0.05,
                                     node_add = 0.3,
                                     node_delete = 0.05,
                                     ),
            crossover=DefaultCrossover(),
            distance=DefaultDistance(compatibility_disjoint = 1.0,
                                     compatibility_weight  = 0.4,
                                     ),
            output_transform=ACT.tanh, # Tanh output activation
            input_transform=None,


                        ),
        max_stagnation =  50,
        species_elitism = 7,
        spawn_number_change_rate= 0.5,
        genome_elitism  = 7,
        min_species_size = 1,
        compatibility_threshold = 0.5,
        species_fitness_func = jnp.max, #custom function
        species_number_calculate_by ="rank",
    )

In [9]:
pipeline = Pipeline(
    algorithm  =neatalgorithm,
    problem=SlimeVolleyEnv(
        max_episode_steps=3000,  # Shorter episodes for faster training
        num_episodes=3  # Average over 3 episodes per evaluation
    ),
    generation_limit=600,  # Run for 200 generations
    fitness_target=10000.0,  # Stop if this fitness is reached
    seed=42,
)



In [10]:
state = pipeline.setup()

state, best = pipeline.auto_run(state)

initializing
initializing finished
start compile
compile finished, cost time: 41.009690s
Generation: 1, Cost time: 3582.01ms
 	fitness: valid cnt: 400, max: -5263.5820, min: -8695.9355, mean: -7074.8325, std: 558.6502

	node counts: max: 16, min: 15, mean: 15.30
 	conn counts: max: 38, min: 35, mean: 36.30
 	species: 200, [1, 6, 2, 4, 7, 5, 8, 3, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 166]

Generation: 2, Cost time: 3835.99ms
 	fitness: valid cnt: 400, max: -528

In [ ]:
# show result
pipeline.show(state, best)


SLIMEVOLLEY AGENT PERFORMANCE
Episode 1 - Reward: -1.00, Steps: 3000, Opponent Life: 5.0
  -> New best episode!
Episode 2 - Reward: 0.00, Steps: 3000, Opponent Life: 5.0
  -> New best episode!
Episode 3 - Reward: -4.00, Steps: 3000, Opponent Life: 5.0
Episode 4 - Reward: 1.00, Steps: 3000, Opponent Life: 4.0
  -> New best episode!
Episode 5 - Reward: 3.00, Steps: 3000, Opponent Life: 2.0


In [ ]:

network = neatalgorithm.genome.network_dict(state, *best)
neatalgorithm.genome.visualize(network, save_path="./volleyball_network.png")

In [ ]:
github_path = "First/NEAT_JAX/Custom_TensorNEAT.ipynb"